# YouTube Dataset Builder for Numeric Inference

This notebook collects video data (titles and views) from specific YouTube channels to build a dataset for predicting video popularity.

## 1. Setup and Authentication

Install the YouTube API client and mount Google Drive.

In [7]:
!pip install -q google-api-python-client

import os
import json
import time
from datetime import datetime
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import drive

try:
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab or drive mount failed.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration

Set up your API key and paths.

In [8]:
# Configuration
from google.colab import userdata

# Retrieve the secret by the name you defined in the sidebar
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')

# Now you can use YOUTUBE_API_KEY in your requests
print("API Key retrieved successfully.")

INPUT_CHANNELS_PATH = '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channels_structured.json'
OUTPUT_DATA_PATH = '/content/drive/MyDrive/Graphiko/exports/base_data/latest/channels_structured_v2.json'
CACHE_DIR = '/content/drive/MyDrive/youtube_api_cache/'
MAX_VIDEOS_PER_CHANNEL = 1000

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR, exist_ok=True)

youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

API Key retrieved successfully.


## 3. Utility Functions

Functions to interact with the YouTube API efficiently.

In [9]:
def get_longform_playlist_id(channel_id):
    """Derive the 'Long-form' playlist ID from a Channel ID."""
    if channel_id.startswith('UC'):
        return 'UULF' + channel_id[2:]
    return None

def fetch_video_ids(playlist_id, max_results=1000):
    """Fetch video IDs from a playlist using playlistItems.list."""
    video_ids = []
    next_page_token = None

    while len(video_ids) < max_results:
        request = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=min(50, max_results - len(video_ids)),
            pageToken=next_page_token
        )
        try:
            response = request.execute()
            video_ids.extend([item['contentDetails']['videoId'] for item in response.get('items', [])])
            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
        except HttpError as e:
            print(f"An error occurred: {e}")
            break

    return video_ids

def fetch_video_details(video_ids):
    """Fetch video details (title, stats, publishedAt) in batches of 50."""
    all_details = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        request = youtube.videos().list(
            part="snippet,statistics",
            id=",".join(batch)
        )
        try:
            response = request.execute()
            for item in response.get('items', []):
                all_details.append({
                    'id': item['id'],
                    'title': item['snippet']['title'],
                    'views': int(item['statistics'].get('viewCount', 0)),
                    'publishedAt': item['snippet']['publishedAt']
                })
        except HttpError as e:
            print(f"An error occurred: {e}")

    return all_details

## 4. Main Collection Loop

Iterate through channels, fetch data, and cache results.

In [10]:
def load_channels():
    with open(INPUT_CHANNELS_PATH, 'r') as f:
        return json.load(f)

def get_cache_path(channel_id):
    return os.path.join(CACHE_DIR, f"{channel_id}.json")

def save_to_cache(channel_id, data):
    with open(get_cache_path(channel_id), 'w') as f:
        json.dump(data, f)

def load_from_cache(channel_id):
    path = get_cache_path(channel_id)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return None

channels_data = load_channels()
final_dataset = []

for channel in channels_data:
    channel_id = channel['channel_id']
    channel_name = channel['channel_name']
    print(f"Processing channel: {channel_name} ({channel_id})")

    cached_videos = load_from_cache(channel_id)
    if cached_videos:
        print(f"  Loaded {len(cached_videos)} videos from cache.")
        final_dataset.append({
            'channel_id': channel_id,
            'channel_name': channel_name,
            'videos': cached_videos
        })
        continue

    playlist_id = get_longform_playlist_id(channel_id)
    if not playlist_id:
        print(f"  Could not derive playlist ID for {channel_id}")
        continue

    print(f"  Fetching video IDs from playlist {playlist_id}...")
    video_ids = fetch_video_ids(playlist_id, MAX_VIDEOS_PER_CHANNEL)

    print(f"  Fetching details for {len(video_ids)} videos...")
    videos_details = fetch_video_details(video_ids)

    save_to_cache(channel_id, videos_details)
    final_dataset.append({
        'channel_id': channel_id,
        'channel_name': channel_name,
        'videos': videos_details
    })
    print(f"  Done. Collected {len(videos_details)} videos.")

# Save final dataset
with open(OUTPUT_DATA_PATH, 'w') as f:
    json.dump(final_dataset, f, indent=2)
print(f"\nFinal dataset saved to {OUTPUT_DATA_PATH}")

Processing channel: Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew)
  Fetching video IDs from playlist UULF-yRDvpR99LUc5l7i7jLzew...
  Fetching details for 45 videos...
  Done. Collected 45 videos.
Processing channel: The Prof G Pod – Scott Galloway (UC1E1SVcVyU3ntWMSQEp38Yw)
  Fetching video IDs from playlist UULF1E1SVcVyU3ntWMSQEp38Yw...
  Fetching details for 526 videos...
  Done. Collected 526 videos.
Processing channel: Asianometry (UC1LpsuAUaKoMzzJSEt5WImw)
  Fetching video IDs from playlist UULF1LpsuAUaKoMzzJSEt5WImw...
  Fetching details for 684 videos...
  Done. Collected 684 videos.
Processing channel: Lenny's Podcast (UC6t1O76G0jYXOAoYCm153dA)
  Fetching video IDs from playlist UULF6t1O76G0jYXOAoYCm153dA...
  Fetching details for 358 videos...
  Done. Collected 358 videos.
Processing channel: a16z (UC9cn0TuPq4dnbTY-CBsm8XA)
  Fetching video IDs from playlist UULF9cn0TuPq4dnbTY-CBsm8XA...
  Fetching details for 1000 videos...
  Done. Collected 1000 videos.
Processing channel: Dan Martell 

## 5. Statistics Reporting

Report statistics per channel and aggregate.

In [11]:
total_videos = 0
print(f"{'Channel Name':<30} | {'Videos':<10} | {'Avg Views':<15}")
print("-" * 60)
for channel in final_dataset:
    count = len(channel['videos'])
    total_videos += count
    avg_views = sum(v['views'] for v in channel['videos']) / count if count > 0 else 0
    print(f"{channel['channel_name']:<30} | {count:<10} | {avg_views:<15,.0f}")

print("-" * 60)
print(f"Total videos across all channels: {total_videos}")

Channel Name                   | Videos     | Avg Views      
------------------------------------------------------------
Bg2 Pod                        | 45         | 83,527         
The Prof G Pod – Scott Galloway | 526        | 60,486         
Asianometry                    | 684        | 211,670        
Lenny's Podcast                | 358        | 46,279         
a16z                           | 1000       | 12,682         
Dan Martell                    | 737        | 158,497        
Patrick Boyle                  | 460        | 370,067        
Real Vision Presents           | 1000       | 13,717         
All-In Podcast                 | 477        | 312,401        
Garry Tan                      | 158        | 54,768         
Valuetainment                  | 1000       | 131,639        
Joe Lonsdale                   | 271        | 23,427         
Tony Robbins                   | 615        | 139,005        
ARK Invest                     | 614        | 42,839         
Network 